# 🎭 Deepfake Audio Detection System

## Project Overview
This notebook implements a complete deepfake audio detection system using the **Fake-or-Real (FoR) Dataset**.

### Objectives:
- Classify audio as **Genuine (Human)** or **Deepfake (AI-Generated)**
- Achieve **≥80% Accuracy** and **≤12% EER**
- Build an end-to-end pipeline from feature extraction to deployment

### Dataset:
- **Fake-or-Real Dataset** from Kaggle
- Using the `for-norm` (normalized) training directory
- ~12,000 balanced samples (real + fake)

### Methodology:
1. Audio Preprocessing (16kHz, normalization, trimming)
2. Feature Extraction (MFCCs, Mel-spectrograms, spectral features)
3. CNN+LSTM Deep Learning Model
4. Comprehensive Evaluation (Accuracy, EER, F1, Confusion Matrix)

## 1. Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install librosa numpy pandas scikit-learn tensorflow matplotlib seaborn soundfile tqdm xgboost

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, roc_curve, auc, classification_report
)
from scipy.optimize import brentq
from scipy.interpolate import interp1d

# XGBoost for comparison
import xgboost as xgb
import joblib

print(f"TensorFlow version: {tf.__version__}")
print(f"Librosa version: {librosa.__version__}")
print("All imports successful!")

In [ ]:
# Configuration
class Config:
    # Paths
    DATA_DIR = Path("../data/for-norm/training")
    REAL_DIR = DATA_DIR / "real"
    FAKE_DIR = DATA_DIR / "fake"
    MODELS_DIR = Path("../models")
    FIGURES_DIR = Path("../figures")
    
    # Audio parameters
    SAMPLE_RATE = 16000
    DURATION = 3  # seconds
    N_SAMPLES = SAMPLE_RATE * DURATION
    
    # Feature extraction
    N_MFCC = 40
    N_MELS = 128
    HOP_LENGTH = 512
    N_FFT = 2048
    
    # Model parameters
    BATCH_SIZE = 32
    EPOCHS = 50
    LEARNING_RATE = 0.001
    PATIENCE = 10
    
    # Data split
    TEST_SIZE = 0.1
    VAL_SIZE = 0.1
    RANDOM_STATE = 42

config = Config()

# Create directories if they don't exist
config.MODELS_DIR.mkdir(parents=True, exist_ok=True)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded successfully!")
print(f"Data directory: {config.DATA_DIR}")

## 2. Data Loading and Exploration

In [ ]:
def load_dataset(data_dir):
    """Load dataset and create dataframe with file paths and labels."""
    real_dir = data_dir / "real"
    fake_dir = data_dir / "fake"
    
    data = []
    
    # Load real samples
    if real_dir.exists():
        for file in tqdm(real_dir.glob("*.wav"), desc="Loading real samples"):
            data.append({"path": str(file), "label": 1, "label_name": "real"})
    
    # Load fake samples
    if fake_dir.exists():
        for file in tqdm(fake_dir.glob("*.wav"), desc="Loading fake samples"):
            data.append({"path": str(file), "label": 0, "label_name": "fake"})
    
    df = pd.DataFrame(data)
    return df.sample(frac=1, random_state=config.RANDOM_STATE).reset_index(drop=True)

# Load dataset
# NOTE: Download dataset from Kaggle and place in ../data/for-norm/training/
# Kaggle link: https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset

if config.DATA_DIR.exists():
    df = load_dataset(config.DATA_DIR)
    print(f"\nDataset loaded: {len(df)} samples")
    print(f"\nClass distribution:")
    print(df["label_name"].value_counts())
else:
    print("=" * 60)
    print("DATASET NOT FOUND!")
    print("=" * 60)
    print("\nPlease download the Fake-or-Real dataset from:")
    print("https://www.kaggle.com/datasets/mohammedabdeldayem/the-fake-or-real-dataset")
    print("\nExtract and place the 'for-norm' folder in: ../data/")
    print("\nExpected structure:")
    print("  data/for-norm/training/real/*.wav")
    print("  data/for-norm/training/fake/*.wav")

In [ ]:
# Display dataset info
if 'df' in locals() and len(df) > 0:
    print("Dataset Info:")
    print(f"  Total samples: {len(df)}")
    print(f"  Real samples: {len(df[df['label'] == 1])}")
    print(f"  Fake samples: {len(df[df['label'] == 0])}")
    print(f"\nSample file paths:")
    print(df.head())

## 3. Audio Preprocessing

In [ ]:
def load_and_preprocess_audio(file_path, sr=config.SAMPLE_RATE, duration=config.DURATION):
    """Load and preprocess audio file."""
    try:
        # Load audio file
        audio, orig_sr = librosa.load(file_path, sr=sr, duration=duration)
        
        # Normalize amplitude
        audio = librosa.util.normalize(audio)
        
        # Trim silence
        audio, _ = librosa.effects.trim(audio, top_db=30)
        
        # Pad or truncate to fixed length
        if len(audio) > config.N_SAMPLES:
            audio = audio[:config.N_SAMPLES]
        else:
            audio = np.pad(audio, (0, config.N_SAMPLES - len(audio)), mode='constant')
        
        return audio
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

print("Audio preprocessing function defined!")

In [ ]:
# Test audio loading with a sample
if 'df' in locals() and len(df) > 0:
    sample_path = df.iloc[0]['path']
    sample_audio = load_and_preprocess_audio(sample_path)
    
    if sample_audio is not None:
        print(f"Sample audio loaded successfully!")
        print(f"  Duration: {len(sample_audio)/config.SAMPLE_RATE:.2f} seconds")
        print(f"  Sample rate: {config.SAMPLE_RATE} Hz")
        print(f"  Samples: {len(sample_audio)}")
        
        # Plot waveform
        plt.figure(figsize=(12, 4))
        librosa.display.waveshow(sample_audio, sr=config.SAMPLE_RATE)
        plt.title('Sample Audio Waveform')
        plt.xlabel('Time (s)')
        plt.ylabel('Amplitude')
        plt.tight_layout()
        plt.show()

## 4. Feature Extraction

In [ ]:
def extract_features(audio, sr=config.SAMPLE_RATE):
    """Extract comprehensive audio features."""
    features = {}
    
    # 1. MFCCs (Mel-Frequency Cepstral Coefficients)
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=config.N_MFCC)
    features['mfccs_mean'] = np.mean(mfccs, axis=1)
    features['mfccs_std'] = np.std(mfccs, axis=1)
    
    # 2. Mel-spectrogram
    mel_spec = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=config.N_MELS)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    features['mel_spec'] = mel_spec_db
    
    # 3. Spectral features
    features['spectral_centroid'] = np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr))
    features['spectral_bandwidth'] = np.mean(librosa.feature.spectral_bandwidth(y=audio, sr=sr))
    features['spectral_rolloff'] = np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr))
    features['spectral_contrast'] = np.mean(librosa.feature.spectral_contrast(y=audio, sr=sr))
    
    # 4. Zero crossing rate
    features['zcr'] = np.mean(librosa.feature.zero_crossing_rate(audio))
    
    # 5. RMS energy
    features['rms'] = np.mean(librosa.feature.rms(y=audio))
    
    # 6. Chroma features
    chroma = librosa.feature.chroma_stft(y=audio, sr=sr)
    features['chroma_mean'] = np.mean(chroma, axis=1)
    
    # 7. Tonnetz (tonal centroid features)
    tonnetz = librosa.feature.tonnetz(y=audio, sr=sr)
    features['tonnetz_mean'] = np.mean(tonnetz, axis=1)
    
    return features

print("Feature extraction function defined!")

In [ ]:
def prepare_dataset(df, max_samples=None):
    """Prepare dataset with extracted features."""
    mfcc_features = []
    mel_features = []
    other_features = []
    labels = []
    
    iterator = tqdm(df.iterrows(), total=len(df), desc="Extracting features")
    if max_samples:
        iterator = tqdm(df.head(max_samples).iterrows(), total=max_samples, desc="Extracting features")
    
    for idx, row in iterator:
        audio = load_and_preprocess_audio(row['path'])
        if audio is not None:
            features = extract_features(audio)
            
            mfcc_features.append(features['mfccs_mean'])
            mel_features.append(features['mel_spec'])
            other_features.append([
                features['spectral_centroid'],
                features['spectral_bandwidth'],
                features['spectral_rolloff'],
                features['spectral_contrast'],
                features['zcr'],
                features['rms']
            ])
            labels.append(row['label'])
    
    return (
        np.array(mfcc_features),
        np.array(mel_features),
        np.array(other_features),
        np.array(labels)
    )

print("Dataset preparation function defined!")

In [ ]:
# Extract features (limit samples for initial testing)
if 'df' in locals() and len(df) > 0:
    # Use subset for faster development (remove max_samples for full dataset)
    MAX_SAMPLES = 1000  # Set to None for full dataset
    
    mfcc_features, mel_features, other_features, labels = prepare_dataset(df, max_samples=MAX_SAMPLES)
    
    print(f"\nFeature extraction complete!")
    print(f"  MFCC features shape: {mfcc_features.shape}")
    print(f"  Mel-spectrogram shape: {mel_features.shape}")
    print(f"  Other features shape: {other_features.shape}")
    print(f"  Labels shape: {labels.shape}")
    print(f"  Label distribution: {np.bincount(labels)}")

## 5. Data Visualization

In [ ]:
def plot_feature_comparison(mfcc_features, labels):
    """Plot feature comparison between real and fake audio."""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    real_features = mfcc_features[labels == 1]
    fake_features = mfcc_features[labels == 0]
    
    # Plot first 6 MFCC coefficients
    for i in range(6):
        ax = axes[i // 3, i % 3]
        ax.hist(real_features[:, i], bins=30, alpha=0.5, label='Real', density=True)
        ax.hist(fake_features[:, i], bins=30, alpha=0.5, label='Fake', density=True)
        ax.set_title(f'MFCC Coefficient {i+1}')
        ax.legend()
    
    plt.suptitle('MFCC Feature Distribution: Real vs Fake', fontsize=14)
    plt.tight_layout()
    plt.savefig(config.FIGURES_DIR / 'mfcc_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

if 'mfcc_features' in locals():
    plot_feature_comparison(mfcc_features, labels)

In [ ]:
def plot_mel_spectrogram_comparison(mel_features, labels):
    """Plot mel-spectrogram comparison."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Real sample
    real_idx = np.where(labels == 1)[0][0]
    librosa.display.specshow(mel_features[real_idx], sr=config.SAMPLE_RATE,
                            x_axis='time', y_axis='mel', ax=axes[0])
    axes[0].set_title('Real Audio - Mel Spectrogram')
    axes[0].set_xlabel('Time')
    axes[0].set_ylabel('Mel Frequency')
    
    # Fake sample
    fake_idx = np.where(labels == 0)[0][0]
    librosa.display.specshow(mel_features[fake_idx], sr=config.SAMPLE_RATE,
                            x_axis='time', y_axis='mel', ax=axes[1])
    axes[1].set_title('Fake Audio - Mel Spectrogram')
    axes[1].set_xlabel('Time')
    axes[1].set_ylabel('Mel Frequency')
    
    plt.suptitle('Mel-Spectrogram Comparison', fontsize=14)
    plt.tight_layout()
    plt.savefig(config.FIGURES_DIR / 'mel_spectrogram_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

if 'mel_features' in locals():
    plot_mel_spectrogram_comparison(mel_features, labels)

## 6. Data Preparation for Training

In [ ]:
# Prepare data for training
if 'mfcc_features' in locals():
    # Combine features for classical ML
    X_classical = np.hstack([mfcc_features, other_features])
    y = labels
    
    # Reshape mel features for CNN+LSTM
    X_cnn = mel_features[..., np.newaxis]  # Add channel dimension
    
    # Split data: 80% train, 10% val, 10% test
    X_train_cnn, X_test_cnn, y_train, y_test = train_test_split(
        X_cnn, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
    )
    
    X_train_cnn, X_val_cnn, y_train, y_val = train_test_split(
        X_train_cnn, y_train, test_size=config.VAL_SIZE/(1-config.TEST_SIZE),
        random_state=config.RANDOM_STATE, stratify=y_train
    )
    
    # Split classical features
    X_train_class, X_test_class, _, _ = train_test_split(
        X_classical, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
    )
    X_train_class, X_val_class, _, _ = train_test_split(
        X_train_class, y_train, test_size=config.VAL_SIZE/(1-config.TEST_SIZE),
        random_state=config.RANDOM_STATE, stratify=y_train
    )
    
    # Scale classical features
    scaler = StandardScaler()
    X_train_class_scaled = scaler.fit_transform(X_train_class)
    X_val_class_scaled = scaler.transform(X_val_class)
    X_test_class_scaled = scaler.transform(X_test_class)
    
    print("Data split complete!")
    print(f"\nCNN+LSTM Data:")
    print(f"  Train: {X_train_cnn.shape}, {y_train.shape}")
    print(f"  Val: {X_val_cnn.shape}, {y_val.shape}")
    print(f"  Test: {X_test_cnn.shape}, {y_test.shape}")
    print(f"\nClassical ML Data:")
    print(f"  Train: {X_train_class_scaled.shape}")
    print(f"  Val: {X_val_class_scaled.shape}")
    print(f"  Test: {X_test_class_scaled.shape}")

## 7. Deep Learning Model (CNN + LSTM)

In [ ]:
def build_cnn_lstm_model(input_shape):
    """Build CNN + LSTM model for deepfake detection."""
    model = models.Sequential([
        # CNN Feature Extractor
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Reshape for LSTM
        layers.Reshape((-1, 128)),
        
        # Temporal Learning
        layers.LSTM(128, return_sequences=False),
        layers.Dropout(0.5),
        
        # Classifier
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config.LEARNING_RATE),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build model
if 'X_train_cnn' in locals():
    input_shape = X_train_cnn.shape[1:]
    model = build_cnn_lstm_model(input_shape)
    model.summary()

In [ ]:
# Train CNN+LSTM model
if 'model' in locals() and 'X_train_cnn' in locals():
    # Callbacks
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=config.PATIENCE,
        restore_best_weights=True,
        verbose=1
    )
    
    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    )
    
    model_checkpoint = callbacks.ModelCheckpoint(
        config.MODELS_DIR / 'deepfake_cnn_lstm_best.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
    
    # Train
    history = model.fit(
        X_train_cnn, y_train,
        validation_data=(X_val_cnn, y_val),
        epochs=config.EPOCHS,
        batch_size=config.BATCH_SIZE,
        callbacks=[early_stop, reduce_lr, model_checkpoint],
        verbose=1
    )
    
    # Save final model
    model.save(config.MODELS_DIR / 'deepfake_cnn_lstm_final.h5')
    print("\nModel saved successfully!")

In [ ]:
# Plot training history
if 'history' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    axes[0].plot(history.history['accuracy'], label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    # Loss
    axes[1].plot(history.history['loss'], label='Train')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.savefig(config.FIGURES_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Classical ML Model (XGBoost)

In [ ]:
# Train XGBoost model
if 'X_train_class_scaled' in locals():
    # Create and train XGBoost classifier
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=config.RANDOM_STATE
    )
    
    xgb_model.fit(
        X_train_class_scaled, y_train,
        eval_set=[(X_val_class_scaled, y_val)],
        verbose=50
    )
    
    # Save model
    joblib.dump(xgb_model, config.MODELS_DIR / 'deepfake_xgboost.pkl')
    joblib.dump(scaler, config.MODELS_DIR / 'scaler.pkl')
    print("\nXGBoost model saved successfully!")

## 9. Model Evaluation

In [ ]:
def calculate_eer(y_true, y_score):
    """Calculate Equal Error Rate (EER)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    eer = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return eer

def evaluate_model(y_true, y_pred, y_prob, model_name):
    """Comprehensive model evaluation."""
    results = {}
    
    # Overall metrics
    results['accuracy'] = accuracy_score(y_true, y_pred)
    results['f1'] = f1_score(y_true, y_pred)
    results['precision'] = precision_score(y_true, y_pred)
    results['recall'] = recall_score(y_true, y_pred)
    
    # EER
    results['eer'] = calculate_eer(y_true, y_prob)
    
    # Per-class accuracy
    cm = confusion_matrix(y_true, y_pred)
    results['per_class_accuracy'] = cm.diagonal() / cm.sum(axis=1)
    
    # ROC-AUC
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    results['roc_auc'] = auc(fpr, tpr)
    results['fpr'] = fpr
    results['tpr'] = tpr
    
    # Print results
    print(f"\n{'='*60}")
    print(f"{model_name} Results")
    print(f"{'='*60}")
    print(f"Overall Accuracy: {results['accuracy']*100:.2f}%")
    print(f"F1 Score: {results['f1']*100:.2f}%")
    print(f"Precision: {results['precision']*100:.2f}%")
    print(f"Recall: {results['recall']*100:.2f}%")
    print(f"EER: {results['eer']*100:.2f}%")
    print(f"ROC-AUC: {results['roc_auc']:.4f}")
    print(f"\nPer-Class Accuracy:")
    print(f"  Fake: {results['per_class_accuracy'][0]*100:.2f}%")
    print(f"  Real: {results['per_class_accuracy'][1]*100:.2f}%")
    print(f"\nConfusion Matrix:")
    print(cm)
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=['Fake', 'Real']))
    
    return results

print("Evaluation functions defined!")

In [ ]:
# Evaluate CNN+LSTM model
if 'model' in locals() and 'X_test_cnn' in locals():
    # Get predictions
    y_prob_cnn = model.predict(X_test_cnn, verbose=0).flatten()
    y_pred_cnn = (y_prob_cnn > 0.5).astype(int)
    
    # Evaluate
    cnn_results = evaluate_model(y_test, y_pred_cnn, y_prob_cnn, "CNN+LSTM")

In [ ]:
# Evaluate XGBoost model
if 'xgb_model' in locals() and 'X_test_class_scaled' in locals():
    # Get predictions
    y_prob_xgb = xgb_model.predict_proba(X_test_class_scaled)[:, 1]
    y_pred_xgb = xgb_model.predict(X_test_class_scaled)
    
    # Evaluate
    xgb_results = evaluate_model(y_test, y_pred_xgb, y_prob_xgb, "XGBoost")

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# CNN+LSTM confusion matrix
if 'cnn_results' in locals():
    cm_cnn = confusion_matrix(y_test, y_pred_cnn)
    sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    axes[0].set_title('CNN+LSTM Confusion Matrix')
    axes[0].set_ylabel('True Label')
    axes[0].set_xlabel('Predicted Label')

# XGBoost confusion matrix
if 'xgb_results' in locals():
    cm_xgb = confusion_matrix(y_test, y_pred_xgb)
    sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues', ax=axes[1],
                xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
    axes[1].set_title('XGBoost Confusion Matrix')
    axes[1].set_ylabel('True Label')
    axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Plot ROC curves
plt.figure(figsize=(8, 6))

if 'cnn_results' in locals():
    plt.plot(cnn_results['fpr'], cnn_results['tpr'],
             label=f"CNN+LSTM (AUC = {cnn_results['roc_auc']:.3f})")

if 'xgb_results' in locals():
    plt.plot(xgb_results['fpr'], xgb_results['tpr'],
             label=f"XGBoost (AUC = {xgb_results['roc_auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(True)
plt.savefig(config.FIGURES_DIR / 'roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Model Comparison and Selection

In [ ]:
# Compare models
comparison_data = []

if 'cnn_results' in locals():
    comparison_data.append({
        'Model': 'CNN+LSTM',
        'Accuracy': cnn_results['accuracy'],
        'F1 Score': cnn_results['f1'],
        'EER': cnn_results['eer'],
        'ROC-AUC': cnn_results['roc_auc'],
        'Fake Accuracy': cnn_results['per_class_accuracy'][0],
        'Real Accuracy': cnn_results['per_class_accuracy'][1]
    })

if 'xgb_results' in locals():
    comparison_data.append({
        'Model': 'XGBoost',
        'Accuracy': xgb_results['accuracy'],
        'F1 Score': xgb_results['f1'],
        'EER': xgb_results['eer'],
        'ROC-AUC': xgb_results['roc_auc'],
        'Fake Accuracy': xgb_results['per_class_accuracy'][0],
        'Real Accuracy': xgb_results['per_class_accuracy'][1]
    })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    comparison_df = comparison_df.set_index('Model')
    
    print("\n" + "="*60)
    print("MODEL COMPARISON")
    print("="*60)
    print(comparison_df.to_string())
    
    # Check if requirements are met
    print("\n" + "="*60)
    print("REQUIREMENTS CHECK")
    print("="*60)
    
    best_model = comparison_df['Accuracy'].idxmax()
    best_results = comparison_df.loc[best_model]
    
    checks = [
        ('Overall Accuracy ≥ 80%', best_results['Accuracy'] >= 0.80),
        ('EER ≤ 12%', best_results['EER'] <= 0.12),
        ('F1 Score ≥ 80%', best_results['F1 Score'] >= 0.80),
        ('Fake Accuracy ≥ 75%', best_results['Fake Accuracy'] >= 0.75),
        ('Real Accuracy ≥ 75%', best_results['Real Accuracy'] >= 0.75)
    ]
    
    for check_name, passed in checks:
        status = "✅ PASS" if passed else "❌ FAIL"
        print(f"  {check_name}: {status}")
    
    print(f"\nBest Model: {best_model}")

## 11. Save Final Model and Artifacts

In [ ]:
# Save all artifacts
print("Saving model artifacts...")

# Save label encoder
le = LabelEncoder()
le.fit(['fake', 'real'])
joblib.dump(le, config.MODELS_DIR / 'label_encoder.pkl')

# Save config
config_dict = {
    'sample_rate': config.SAMPLE_RATE,
    'duration': config.DURATION,
    'n_mfcc': config.N_MFCC,
    'n_mels': config.N_MELS,
    'hop_length': config.HOP_LENGTH,
    'n_fft': config.N_FFT
}
joblib.dump(config_dict, config.MODELS_DIR / 'config.pkl')

print(f"\nAll artifacts saved to: {config.MODELS_DIR}")
print("Files saved:")
for file in config.MODELS_DIR.glob('*'):
    print(f"  - {file.name}")

## 12. Inference Function

In [ ]:
def predict_audio(file_path, model, scaler=None, config_dict=None):
    """Predict if audio is real or fake."""
    # Load and preprocess audio
    audio = load_and_preprocess_audio(file_path)
    if audio is None:
        return None
    
    # Extract features
    features = extract_features(audio)
    
    # For CNN+LSTM model
    if hasattr(model, 'predict') and len(model.input_shape) == 5:  # CNN
        mel_spec = features['mel_spec']
        mel_spec = mel_spec[..., np.newaxis]
        mel_spec = np.expand_dims(mel_spec, axis=0)
        prob = model.predict(mel_spec, verbose=0)[0][0]
    else:  # Classical ML
        mfcc_features = features['mfccs_mean']
        other_features_arr = [
            features['spectral_centroid'],
            features['spectral_bandwidth'],
            features['spectral_rolloff'],
            features['spectral_contrast'],
            features['zcr'],
            features['rms']
        ]
        X = np.hstack([mfcc_features, other_features_arr]).reshape(1, -1)
        if scaler:
            X = scaler.transform(X)
        prob = model.predict_proba(X)[0][1]
    
    # Determine prediction
    prediction = 'real' if prob > 0.5 else 'fake'
    confidence = prob if prob > 0.5 else 1 - prob
    
    return {
        'prediction': prediction,
        'confidence': float(confidence),
        'probability_real': float(prob),
        'probability_fake': float(1 - prob)
    }

print("Inference function defined!")

In [ ]:
# Test inference on sample files
if 'df' in locals() and len(df) > 0 and 'model' in locals():
    # Test with a real sample
    real_sample = df[df['label'] == 1].iloc[0]['path']
    result = predict_audio(real_sample, model)
    print(f"Real sample prediction: {result}")
    
    # Test with a fake sample
    fake_sample = df[df['label'] == 0].iloc[0]['path']
    result = predict_audio(fake_sample, model)
    print(f"Fake sample prediction: {result}")

## 13. Summary

In [ ]:
print("\n" + "="*60)
print("DEEPFAKE AUDIO DETECTION - TRAINING COMPLETE")
print("="*60)
print("\nModels trained and saved:")
print(f"  1. CNN+LSTM: {config.MODELS_DIR / 'deepfake_cnn_lstm_final.h5'}")
print(f"  2. XGBoost: {config.MODELS_DIR / 'deepfake_xgboost.pkl'}")
print(f"\nArtifacts saved:")
print(f"  - Scaler: {config.MODELS_DIR / 'scaler.pkl'}")
print(f"  - Label Encoder: {config.MODELS_DIR / 'label_encoder.pkl'}")
print(f"  - Config: {config.MODELS_DIR / 'config.pkl'}")
print(f"\nFigures saved to: {config.FIGURES_DIR}")
print("\nNext steps:")
print("  1. Run test_audio.py to test with new audio files")
print("  2. Deploy app.py using Streamlit")
print("  3. Generate performance report")